[link text](https://)

In [2]:
#task 1
import pandas as pd
import numpy as np
df=pd.read_csv('bangalore_tech_salaries.csv')
df.info()
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df.duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Employee_ID     1015 non-null   object
 1   Role            1015 non-null   object
 2   years_exp       1015 non-null   int64 
 3   Current_CTC     1015 non-null   object
 4   Previous_CTC    815 non-null    object
 5   Company         1015 non-null   object
 6   company_TYPE    1015 non-null   object
 7   Skills          988 non-null    object
 8   Location        995 non-null    object
 9   Education_Tier  1015 non-null   object
 10  Joining_Year    1015 non-null   int64 
 11  Work_Mode       1015 non-null   object
dtypes: int64(2), object(10)
memory usage: 95.3+ KB
(1015, 12)
Employee_ID       object
Role              object
years_exp          int64
Current_CTC       object
Previous_CTC      object
Company           object
company_TYPE      object
Skills            object
Location      

In [3]:
print("DATASET QUALITY SUMMARY")
print(f"1. The dataset contains {df.shape[0]} employee records and {df.shape[1]} columns.")
print("2. The dataset contains both numerical and categorical features.")
print(f"3. A total of {df.isnull().sum().sum()} missing values were found across the dataset.")
print(f"4. The dataset contains {df.duplicated().sum()} duplicate rows before cleaning.")
print("5. The Role, Company Type, Education Tier and Current CTC columns contain inconsistent formats and require cleaning.")
print("6. Current CTC values appear in multiple formats (LPA, ₹LPA and rupee values), so they will be standardised into LPA.")
print("7. After cleaning, the dataset will be ready for business analysis.")

DATASET QUALITY SUMMARY
1. The dataset contains 1015 employee records and 12 columns.
2. The dataset contains both numerical and categorical features.
3. A total of 247 missing values were found across the dataset.
4. The dataset contains 15 duplicate rows before cleaning.
5. The Role, Company Type, Education Tier and Current CTC columns contain inconsistent formats and require cleaning.
6. Current CTC values appear in multiple formats (LPA, ₹LPA and rupee values), so they will be standardised into LPA.
7. After cleaning, the dataset will be ready for business analysis.


In [4]:
#task 2
# rename the column
df.columns = df.columns.str.strip()
df = df.rename(columns={
    "Employee_ID": "employee_id",
    "Role": "role",
    "Current_CTC": "current_ctc",
    "Previous_CTC": "previous_ctc",
    "Company": "company",
    "company_TYPE": "company_type",
    "Skills": "skills",
    "Location": "location",
    "Education_Tier": "education_tier",
    "Joining_Year": "joining_year",
    "Work_Mode": "work_mode"
})
print(df.columns)

Index(['employee_id', 'role', 'years_exp', 'current_ctc', 'previous_ctc',
       'company', 'company_type', 'skills', 'location', 'education_tier',
       'joining_year', 'work_mode'],
      dtype='object')


In [5]:
#roll standerdisation
role_mapping = {
    "Backend Dev":"SDE Backend",
    "Backend Developer":"SDE Backend",
    "Backend Engineer":"SDE Backend",
    "BE":"SDE Backend",
    "SDE-Backend":"SDE Backend",

    "Frontend Dev":"SDE Frontend",
    "Frontend Developer":"SDE Frontend",
    "Frontend Engineer":"SDE Frontend",
    "FE":"SDE Frontend",
    "SDE-Frontend":"SDE Frontend",

    "FullStack":"SDE Full-Stack",
    "Full Stack Engineer":"SDE Full-Stack",
    "Fullstack Dev":"SDE Full-Stack",
    "SDE FS":"SDE Full-Stack",

    "DS":"Data Scientist",
    "Data Science Engineer":"Data Scientist",

    "DA":"Data Analyst",
    "BI Analyst":"Data Analyst",
    "Analytics Engineer":"Data Analyst",

    "BA":"Business Analyst",
    "Business Systems Analyst":"Business Analyst",

    "Designer":"UI/UX",
    "UI Designer":"UI/UX",
    "UX Designer":"UI/UX",
    "Product Designer":"UI/UX",

    "PM":"Product Manager",
    "Sr PM":"Product Manager",
    "Product Lead":"Product Manager",

    "DevOps":"DevOps Engineer",

    "SRE":"Site Reliability Engineer"
}

df["role_clean"] = df["role"].str.strip().replace(role_mapping)

In [6]:
#ctc cleaning
def convert_lpa(value):
    if pd.isna(value):
        return value
    value = str(value).strip()
    if "LPA" in value:
        value = value.replace("₹","")
        value = value.replace("LPA","")
        value = value.strip()
        return float(value)
    value = value.replace(",","")
    number = pd.to_numeric(value, errors="coerce")
    if number > 1000:
        return number / 100000
    return number
df["current_ctc"] = df["current_ctc"].apply(convert_lpa)
df["previous_ctc"] = df["previous_ctc"].apply(convert_lpa)
df["current_ctc"] = pd.to_numeric(df["current_ctc"], errors="coerce")
df["previous_ctc"] = pd.to_numeric(df["previous_ctc"], errors="coerce")

print(df[["current_ctc", "previous_ctc"]].dtypes)

current_ctc     float64
previous_ctc    float64
dtype: object


In [7]:
#company type
df["company_type"] = df["company_type"].str.strip().str.lower()
company_mapping = {
    "mnc": "MNC",
    "unicorn": "Unicorn",
    "mid-size": "Mid-size",
    "early-stage": "Early-stage"
}
df["company_type_clean"] = df["company_type"].replace(company_mapping)
print(df["company_type_clean"].unique())

['Unicorn' 'MNC' 'Mid-size' 'Early-stage']


In [8]:
#Education Tier
education_mapping = {
    "Tier-1": "Tier 1",
    "Tier 1": "Tier 1",
    "T1": "Tier 1",
    "1": "Tier 1",

    "Tier-2": "Tier 2",
    "Tier 2": "Tier 2",
    "T2": "Tier 2",
    "2": "Tier 2",

    "Tier-3": "Tier 3",
    "Tier 3": "Tier 3",
    "T3": "Tier 3",
    "3": "Tier 3"
}

df["education_tier_clean"] = (
    df["education_tier"]
      .astype(str)
      .str.strip()
      .replace(education_mapping)
)

print(df["education_tier_clean"].value_counts())
df["education_tier_clean"] = df["education_tier"].astype(str).str.strip()

df.loc[df["education_tier_clean"].isin(["Tier-1", "Tier 1", "T1", "1"]), "education_tier_clean"] = "Tier 1"

df.loc[df["education_tier_clean"].isin(["Tier-2", "Tier 2", "T2", "2"]), "education_tier_clean"] = "Tier 2"

df.loc[df["education_tier_clean"].isin(["Tier-3", "Tier 3", "T3", "3"]), "education_tier_clean"] = "Tier 3"

print(df["education_tier_clean"].value_counts())

education_tier_clean
Tier 2    510
Tier 3    309
Tier 1    196
Name: count, dtype: int64
education_tier_clean
Tier 2    510
Tier 3    309
Tier 1    196
Name: count, dtype: int64


In [9]:
#Handle Missing Values
df["skills"] = df["skills"].fillna("Not Specified")
df["location"] = df["location"].fillna("Unknown")

In [10]:
#Remove Duplicates
print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after:", df.duplicated().sum())
print(df.dtypes)
print(df["role_clean"].value_counts())
print(df["company_type_clean"].value_counts())
print(df["education_tier_clean"].value_counts())

Duplicates before: 15
Duplicates after: 0
employee_id              object
role                     object
years_exp                 int64
current_ctc             float64
previous_ctc            float64
company                  object
company_type             object
skills                   object
location                 object
education_tier           object
joining_year              int64
work_mode                object
role_clean               object
company_type_clean       object
education_tier_clean     object
dtype: object
role_clean
SDE Backend                  114
UI/UX                        113
Product Manager              112
Business Analyst             108
SDE Frontend                 105
Data Analyst                 100
SDE Full-Stack                96
Data Scientist                94
Site Reliability Engineer     61
DevOps Engineer               46
ML Engineer                   31
Infra Engineer                20
Name: count, dtype: int64
company_type_clean
MNC         

In [11]:
#task 3.1
#role_salary by using groupby
role_salary = df.groupby("role_clean")["current_ctc"].agg(
    median_ctc="median",
    mean_ctc="mean",
    min_ctc="min",
    max_ctc="max"
)

role_salary = role_salary.sort_values(by="median_ctc", ascending=False)

print(role_salary)

print("\nHighest Paying Role:", role_salary.index[0])
print("Lowest Paying Role:", role_salary.index[-1])

                           median_ctc   mean_ctc  min_ctc  max_ctc
role_clean                                                        
Product Manager                 31.30  34.131250     10.8     80.1
ML Engineer                     27.50  30.335484     10.0     64.5
Data Scientist                  24.15  26.670213     10.9     75.6
Infra Engineer                  23.45  22.610000      9.2     45.5
Site Reliability Engineer       23.30  23.767213      8.8     55.0
SDE Full-Stack                  22.35  25.409375      8.9     71.7
SDE Frontend                    21.20  22.193333      6.7     84.4
SDE Backend                     21.10  23.284210      7.9     55.1
Business Analyst                19.90  21.873148      6.8     52.7
DevOps Engineer                 19.45  22.891304      9.7     60.3
UI/UX                           18.90  20.968142      6.2     63.3
Data Analyst                    16.90  17.874000      5.2     43.4

Highest Paying Role: Product Manager
Lowest Paying Role: Data

### Insight

Product Manager has the highest median CTC of **31.30 LPA**, while Data Analyst has the lowest median CTC of **16.90 LPA**. Roles such as SDE Full-Stack and SDE Backend have mean CTC values higher than their median CTC, indicating the presence of a few high-salary outliers.

In [12]:
#task 3.2
# Filter SDE Backend employees
backend_df = df[df["role_clean"] == "SDE Backend"].copy()
backend_df["exp_band"] = pd.cut(
    backend_df["years_exp"],
    bins=[-1, 1, 3, 5, backend_df["years_exp"].max()],
    labels=["0-1", "2-3", "4-5", "6+"]
)
# Calculate median CTC using groupby()
experience_ctc = backend_df.groupby("exp_band")["current_ctc"].median()
print(experience_ctc)

exp_band
0-1    11.65
2-3    20.00
4-5    25.85
6+     40.40
Name: current_ctc, dtype: float64


/tmp/ipykernel_670/2770694483.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  experience_ctc = backend_df.groupby("exp_band")["current_ctc"].median()


### Insight

The median CTC for SDE Backend professionals increases from **11.65 LPA** (0–1 years) to **20.00 LPA** (2–3 years), **25.85 LPA** (4–5 years), and **40.40 LPA** (6+ years). This shows that gaining backend experience significantly increases earning potential, with the highest salaries achieved after 6+ years of experience.

In [13]:
#task 3.3
# AI-assisted: Compared median CTC of SDE professionals with and without selected skills.

# Filter SDE roles
sde_df = df[df["role_clean"].isin([
    "SDE Backend",
    "SDE Frontend",
    "SDE Full-Stack"
])]

# Select four skills
skills = ["AWS", "ML", "System Design", "Kubernetes"]

print("Skill Premium for SDE Professionals\n")

for skill in skills:

    has_skill = sde_df[sde_df["skills"].str.contains(skill, case=False, na=False)]

    no_skill = sde_df[~sde_df["skills"].str.contains(skill, case=False, na=False)]

    median_has = has_skill["current_ctc"].median()
    median_no = no_skill["current_ctc"].median()

    premium = median_has - median_no

    print(f"{skill}")
    print(f"Median CTC (Has {skill})    : {median_has:.2f} LPA")
    print(f"Median CTC (No {skill})     : {median_no:.2f} LPA")
    print(f"Skill Premium               : {premium:.2f} LPA")
    print("-" * 40)

Skill Premium for SDE Professionals

AWS
Median CTC (Has AWS)    : 20.95 LPA
Median CTC (No AWS)     : 21.90 LPA
Skill Premium               : -0.95 LPA
----------------------------------------
ML
Median CTC (Has ML)    : 23.95 LPA
Median CTC (No ML)     : 21.30 LPA
Skill Premium               : 2.65 LPA
----------------------------------------
System Design
Median CTC (Has System Design)    : 25.30 LPA
Median CTC (No System Design)     : 21.00 LPA
Skill Premium               : 4.30 LPA
----------------------------------------
Kubernetes
Median CTC (Has Kubernetes)    : 23.20 LPA
Median CTC (No Kubernetes)     : 21.50 LPA
Skill Premium               : 1.70 LPA
----------------------------------------


### Insight

Among the four selected skills, **System Design** provides the highest salary premium, increasing the median CTC from **21.00 LPA** to **25.30 LPA**, a gain of **4.30 LPA**. Students preparing for SDE roles should prioritise learning System Design to improve their earning potential.

In [14]:
#task 3.4
# Filter SDE Backend professionals
backend_df = df[df["role_clean"] == "SDE Backend"]

# Calculate median CTC by company type
company_ctc = backend_df.groupby("company_type_clean")["current_ctc"].median()

print("Median CTC by Company Type (LPA)\n")
print(company_ctc)

# Calculate Unicorn premium over MNC
unicorn_ctc = company_ctc["Unicorn"]
mnc_ctc = company_ctc["MNC"]

premium = ((unicorn_ctc - mnc_ctc) / mnc_ctc) * 100

print("\nUnicorn Premium over MNC = {:.2f}%".format(premium))

Median CTC by Company Type (LPA)

company_type_clean
Early-stage    18.60
MNC            20.45
Mid-size       19.50
Unicorn        27.35
Name: current_ctc, dtype: float64

Unicorn Premium over MNC = 33.74%


# Task 3.4 – Company-Type Premium for SDE Backend

Compare the median Current CTC of SDE Backend professionals across different company types and calculate the percentage salary premium of Unicorn companies over MNCs.

In [15]:
# task 3.5
df["exp_band"] = pd.cut(
    df["years_exp"],
    bins=[-1, 1, 3, 5, 100],
    labels=["0-1", "2-3", "4-5", "6+"]
)

# Count employees in each group
group_size = df.groupby(
    ["role_clean", "company_type_clean", "exp_band"],
    observed=False
)["current_ctc"].transform("count")

# Keep only groups with at least 10 members
filtered_df = df[group_size >= 10].copy()

# Calculate group median CTC
filtered_df["group_median"] = filtered_df.groupby(
    ["role_clean", "company_type_clean", "exp_band"],
    observed=False
)["current_ctc"].transform("median")

# Calculate salary gap
filtered_df["gap"] = (
    filtered_df["current_ctc"] - filtered_df["group_median"]
)

# Top 10 most underpaid professionals
underpaid = filtered_df.sort_values("gap").head(10)

# Display required columns
print(
    underpaid[
        [
            "employee_id",
            "role_clean",
            "company_type_clean",
            "years_exp",
            "current_ctc",
            "group_median",
            "gap"
        ]
    ]
)

    employee_id        role_clean company_type_clean  years_exp  current_ctc  \
904     BLR0925       SDE Backend                MNC          4         19.4   
250     BLR0072       SDE Backend                MNC          4         19.5   
249     BLR0733  Business Analyst                MNC          7         31.1   
728     BLR0301   Product Manager                MNC          2         24.7   
386     BLR0453  Business Analyst                MNC          6         32.1   
753     BLR0611   Product Manager                MNC          2         25.1   
711     BLR0451      SDE Frontend                MNC          4         19.5   
688     BLR0203       SDE Backend                MNC          4         21.3   
858     BLR0033      SDE Frontend                MNC          4         19.6   
538     BLR0486   Product Manager                MNC          2         25.7   

     group_median   gap  
904         27.30 -7.90  
250         27.30 -7.80  
249         38.50 -7.40  
728         31.

### Insight

The most underpaid professional is **Employee BLR0925 (SDE Backend, MNC)**, earning **19.40 LPA**, which is **7.90 LPA below** the group median of **27.30 LPA**. This analysis highlights employees whose salaries are significantly below their peers with the same role, company type and experience, helping identify opportunities for salary negotiation or better-paying roles.

In [16]:
#task 4
# AI-assisted: Generated the final ASCII formatted salary analysis report.

print("=" * 70)
print("                 BANGALORE TECH SALARY DECODER")
print("              Built by Pallavi C N | Live Project")
print("=" * 70)

print("\nDataset : 1,000 Bengaluru tech professionals")
print("Period  : 2024 employment snapshot")

print("\n----- MEDIAN CTC BY ROLE (in LPA) -----")
print(f"{'Product Manager':<30}{31.30:>6.2f}")
print(f"{'ML Engineer':<30}{27.50:>6.2f}")
print(f"{'Data Scientist':<30}{24.15:>6.2f}")
print(f"{'Infra Engineer':<30}{23.45:>6.2f}")
print(f"{'Site Reliability Engineer':<30}{23.30:>6.2f}")
print(f"{'SDE Full-Stack':<30}{22.35:>6.2f}")
print(f"{'SDE Frontend':<30}{21.20:>6.2f}")
print(f"{'SDE Backend':<30}{21.10:>6.2f}")
print(f"{'Business Analyst':<30}{19.90:>6.2f}")
print(f"{'DevOps Engineer':<30}{19.45:>6.2f}")
print(f"{'UI/UX':<30}{18.90:>6.2f}")
print(f"{'Data Analyst':<30}{16.90:>6.2f}")

print("\n----- SDE BACKEND CTC BY EXPERIENCE BAND -----")
print(f"{'0 to 1 years':<20}{11.65:>6.2f} LPA")
print(f"{'2 to 3 years':<20}{20.00:>6.2f} LPA")
print(f"{'4 to 5 years':<20}{25.85:>6.2f} LPA")
print(f"{'6+ years':<20}{40.40:>6.2f} LPA")
print("\n----- SKILL PREMIUM FOR SDEs (Median CTC) -----")
print(f"{'Skill':<18}{'With':>10}{'Without':>12}{'Premium':>12}")
print("-" * 52)
print(f"{'AWS':<18}{20.95:>10.2f}{21.90:>12.2f}{-0.95:>12.2f}")
print(f"{'ML':<18}{23.95:>10.2f}{21.30:>12.2f}{2.65:>12.2f}")
print(f"{'System Design':<18}{25.30:>10.2f}{21.00:>12.2f}{4.30:>12.2f}")
print(f"{'Kubernetes':<18}{23.20:>10.2f}{21.50:>12.2f}{1.70:>12.2f}")

print("\n----- COMPANY-TYPE PREMIUM (SDE Backend) -----")
print(f"{'Unicorn':<20}{27.35:.2f} LPA")
print(f"{'MNC':<20}{20.45:.2f} LPA")
print(f"{'Mid-size':<20}{19.50:.2f} LPA")
print(f"{'Early-stage':<20}{18.60:.2f} LPA")
print(f"\nUnicorn Premium over MNC : {33.74:.2f}%")

print("\n----- TOP 5 MOST UNDERPAID PROFESSIONALS -----")
print(f"{'Employee ID':<12}{'Role':<22}{'Type':<12}{'Exp':<6}{'Gap'}")
print("-" * 65)
print(f"{'BLR0925':<12}{'SDE Backend':<22}{'MNC':<12}{'4 yrs':<6}{'-7.90 LPA'}")
print(f"{'BLR0072':<12}{'SDE Backend':<22}{'MNC':<12}{'4 yrs':<6}{'-7.80 LPA'}")
print(f"{'BLR0733':<12}{'Business Analyst':<22}{'MNC':<12}{'7 yrs':<6}{'-7.40 LPA'}")
print(f"{'BLR0301':<12}{'Product Manager':<22}{'MNC':<12}{'2 yrs':<6}{'-6.75 LPA'}")
print(f"{'BLR0453':<12}{'Business Analyst':<22}{'MNC':<12}{'6 yrs':<6}{'-6.40 LPA'}")

print("\n" + "=" * 70)
print("                    END OF REPORT")
print("=" * 70)


                 BANGALORE TECH SALARY DECODER
              Built by Pallavi C N | Live Project

Dataset : 1,000 Bengaluru tech professionals
Period  : 2024 employment snapshot

----- MEDIAN CTC BY ROLE (in LPA) -----
Product Manager                31.30
ML Engineer                    27.50
Data Scientist                 24.15
Infra Engineer                 23.45
Site Reliability Engineer      23.30
SDE Full-Stack                 22.35
SDE Frontend                   21.20
SDE Backend                    21.10
Business Analyst               19.90
DevOps Engineer                19.45
UI/UX                          18.90
Data Analyst                   16.90

----- SDE BACKEND CTC BY EXPERIENCE BAND -----
0 to 1 years         11.65 LPA
2 to 3 years         20.00 LPA
4 to 5 years         25.85 LPA
6+ years             40.40 LPA

----- SKILL PREMIUM FOR SDEs (Median CTC) -----
Skill                   With     Without     Premium
----------------------------------------------------
AWS       

# KEY INSIGHTS

1. **Product Manager has the highest median CTC (31.30 LPA), which is 14.40 LPA higher than the median CTC of Data Analyst (16.90 LPA).** Students who begin their careers in technical roles can increase their long-term earning potential by developing product management skills and targeting PM roles later in their careers.

2. **Among SDE Backend professionals, the median CTC rises from 11.65 LPA for employees with 0–1 years of experience to 40.40 LPA for those with 6+ years, an increase of 246.8%.** Students should focus on building long-term backend expertise because salary growth accelerates significantly with experience.

3. **System Design is the highest-paying skill among the four analysed skills, increasing the median CTC from 21.00 LPA to 25.30 LPA (+4.30 LPA), whereas AWS shows a negative premium of 0.95 LPA in this dataset.** Students preparing for SDE interviews should prioritise System Design over simply collecting cloud certifications if their goal is to maximise salary.